# Tugas 1 Analisis Big Data — 01. Data Profiling
**Nama Dataset:** NASA FIRMS VIIRS S-NPP 375m (Titik Panas Indonesia 2019-2024)

Notebook ini bertujuan untuk melakukan profiling awal dataset titik panas (hotspot) di Indonesia dari tahun 2019 hingga 2024.
Hal yang diperiksa:
1. Pemenuhan syarat tugas (minimal 500 MB atau lebih dari 1.000.000 baris).
2. Kualitas data (kelengkapan/missing values, duplikasi, sebaran koordinat wilayah, dan rentang waktu).
3. Profiling menggunakan Polars Lazy (`scan_csv`) dan DuckDB (`SUMMARIZE`).
4. Menyimpan data gabungan ke format Parquet di folder `data/processed/`.

## 1. Setup Library dan Path Data

In [1]:
import glob
import os
from pathlib import Path

import polars as pl
import duckdb

# Deteksi otomatis path data (bisa dijalankan dari root repo maupun dari dalam folder notebooks/)
if Path("data/raw").is_dir():
    RAW_DIR = Path("data/raw")
    PROCESSED_DIR = Path("data/processed")
else:
    RAW_DIR = Path("../data/raw")
    PROCESSED_DIR = Path("../data/processed")

DATA_GLOB = str(RAW_DIR / "*.csv")

# Batas kasar koordinat Indonesia (perkiraan bounding box: bujur 95-141, lintang -11 s.d. 6)
LON_MIN, LON_MAX = 95.0, 141.0
LAT_MIN, LAT_MAX = -11.0, 6.0

# Cari semua file CSV tahunan
files = sorted(glob.glob(DATA_GLOB))
assert len(files) > 0, f"Tidak ada file CSV di {DATA_GLOB}. Pastikan file sudah ada di folder data/raw/."

print(f"Polars versi : {pl.__version__}")
print(f"DuckDB versi : {duckdb.__version__}")
print(f"Path data raw: {RAW_DIR.resolve()}")
print(f"Total file   : {len(files)} file ditemukan")
for f in files:
    print(" -", os.path.basename(f))

Polars versi : 1.44.2
DuckDB versi : 1.5.6
Path data raw: C:\Users\ASUS TUFF\KuliahC\Big Data\Tugas1\tugas1-abd-2026\data\raw
Total file   : 6 file ditemukan
 - viirs-snpp_2019_Indonesia.csv
 - viirs-snpp_2020_Indonesia.csv
 - viirs-snpp_2021_Indonesia.csv
 - viirs-snpp_2022_Indonesia.csv
 - viirs-snpp_2023_Indonesia.csv
 - viirs-snpp_2024_Indonesia.csv


## 2. Inventaris Ukuran dan Baris per File

In [2]:
# Hitung ukuran file dan jumlah baris per file CSV
daftar_inventaris = []

for f in files:
    nama_file = os.path.basename(f)
    ukuran_mb = os.path.getsize(f) / (1024 * 1024)
    # Hitung baris secara lazy dengan Polars scan_csv
    jml_baris = pl.scan_csv(f).select(pl.len()).collect().item()
    
    daftar_inventaris.append({
        "nama_file": nama_file,
        "ukuran_mb": round(ukuran_mb, 2),
        "jumlah_baris": jml_baris
    })

df_inventaris = pl.DataFrame(daftar_inventaris)
print(df_inventaris)

total_ukuran_mb = df_inventaris["ukuran_mb"].sum()
total_baris_semua = df_inventaris["jumlah_baris"].sum()

print(f"\nTotal Ukuran: {total_ukuran_mb:.2f} MB")
print(f"Total Baris : {total_baris_semua:,} baris")

shape: (6, 3)
┌───────────────────────────────┬───────────┬──────────────┐
│ nama_file                     ┆ ukuran_mb ┆ jumlah_baris │
│ ---                           ┆ ---       ┆ ---          │
│ str                           ┆ f64       ┆ i64          │
╞═══════════════════════════════╪═══════════╪══════════════╡
│ viirs-snpp_2019_Indonesia.csv ┆ 33.01     ┆ 437281       │
│ viirs-snpp_2020_Indonesia.csv ┆ 6.9       ┆ 91536        │
│ viirs-snpp_2021_Indonesia.csv ┆ 5.15      ┆ 68325        │
│ viirs-snpp_2022_Indonesia.csv ┆ 4.31      ┆ 57220        │
│ viirs-snpp_2023_Indonesia.csv ┆ 19.14     ┆ 253388       │
│ viirs-snpp_2024_Indonesia.csv ┆ 7.86      ┆ 104186       │
└───────────────────────────────┴───────────┴──────────────┘



Total Ukuran: 76.37 MB
Total Baris : 1,011,936 baris


## 3. Konsistensi Kolom Antar File CSV

In [3]:
# Ambil nama kolom file pertama sebagai acuan
kolom_acuan = pl.scan_csv(files[0]).collect_schema().names()
print(f"Kolom acuan ({os.path.basename(files[0])}):")
print(kolom_acuan)
print()

# Periksa apakah file lainnya memiliki kolom yang sama persis
semua_sama = True
for f in files:
    kolom_file = pl.scan_csv(f).collect_schema().names()
    if kolom_file == kolom_acuan:
        print(f"[OK] {os.path.basename(f)}: 15 kolom cocok")
    else:
        print(f"[BEDA] {os.path.basename(f)} ada ketidakcocokan kolom!")
        semua_sama = False

if semua_sama:
    print("\nKesimpulan: Semua file CSV memiliki struktur kolom yang seragam.")

Kolom acuan (viirs-snpp_2019_Indonesia.csv):
['latitude', 'longitude', 'bright_ti4', 'scan', 'track', 'acq_date', 'acq_time', 'satellite', 'instrument', 'confidence', 'version', 'bright_ti5', 'frp', 'daynight', 'type']

[OK] viirs-snpp_2019_Indonesia.csv: 15 kolom cocok
[OK] viirs-snpp_2020_Indonesia.csv: 15 kolom cocok
[OK] viirs-snpp_2021_Indonesia.csv: 15 kolom cocok
[OK] viirs-snpp_2022_Indonesia.csv: 15 kolom cocok
[OK] viirs-snpp_2023_Indonesia.csv: 15 kolom cocok
[OK] viirs-snpp_2024_Indonesia.csv: 15 kolom cocok

Kesimpulan: Semua file CSV memiliki struktur kolom yang seragam.


## 4. Tipe Data Kolom: Polars vs DuckDB

In [4]:
# Inisialisasi Polars LazyFrame gabungan untuk seluruh CSV
lf = pl.scan_csv(DATA_GLOB)
schema_polars = lf.collect_schema()

# Ambil tipe data dari DuckDB menggunakan DESCRIBE
desc_duckdb = duckdb.sql(f"DESCRIBE SELECT * FROM read_csv_auto('{DATA_GLOB}', union_by_name=true)").fetchall()
tipe_duckdb = {row[0]: row[1] for row in desc_duckdb}

# Tampilkan perbandingan tipe data dalam tabel sederhana
list_kolom = []
list_polars = []
list_duckdb = []

for col in schema_polars.names():
    list_kolom.append(col)
    list_polars.append(str(schema_polars[col]))
    list_duckdb.append(tipe_duckdb.get(col, "-"))

df_tipe = pl.DataFrame({
    "Nama Kolom": list_kolom,
    "Tipe Polars": list_polars,
    "Tipe DuckDB": list_duckdb
})
df_tipe

Nama Kolom,Tipe Polars,Tipe DuckDB
str,str,str
"""latitude""","""Float64""","""DOUBLE"""
"""longitude""","""Float64""","""DOUBLE"""
"""bright_ti4""","""Float64""","""DOUBLE"""
"""scan""","""Float64""","""DOUBLE"""
"""track""","""Float64""","""DOUBLE"""
…,…,…
"""version""","""Int64""","""BIGINT"""
"""bright_ti5""","""Float64""","""DOUBLE"""
"""frp""","""Float64""","""DOUBLE"""


## 5. Pengecekan Syarat Dataset Tugas

In [5]:
# Syarat dataset: Minimal 500 MB ATAU lebih dari 1.000.000 baris
total_baris = lf.select(pl.len()).collect().item()
total_kolom = len(schema_polars.names())

syarat_baris = total_baris > 1_000_000
syarat_mb = total_ukuran_mb >= 500.0

print(f"Total baris data : {total_baris:,}")
print(f"Total ukuran CSV : {total_ukuran_mb:.2f} MB")
print(f"Total kolom      : {total_kolom}")
print("-" * 45)
print(f"1. Syarat > 1 Juta Baris : {'LOLOS' if syarat_baris else 'TIDAK'} ({total_baris:,} baris)")
print(f"2. Syarat >= 500 MB      : {'LOLOS' if syarat_mb else 'TIDAK'} ({total_ukuran_mb:.2f} MB)")
print("-" * 45)
if syarat_baris or syarat_mb:
    print("STATUS TUGAS: TERPENUHI (karena salah satu syarat sudah terpenuhi)")
else:
    print("STATUS TUGAS: BELUM TERPENUHI")

Total baris data : 1,011,936
Total ukuran CSV : 76.37 MB
Total kolom      : 15
---------------------------------------------
1. Syarat > 1 Juta Baris : LOLOS (1,011,936 baris)
2. Syarat >= 500 MB      : TIDAK (76.37 MB)
---------------------------------------------
STATUS TUGAS: TERPENUHI (karena salah satu syarat sudah terpenuhi)


## 6. Contoh Sampel Data (5 Baris)

In [6]:
# 5 baris pertama menggunakan Polars head
lf.head(5).collect()

latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight,type
f64,f64,f64,f64,f64,str,i64,str,str,str,i64,f64,f64,str,i64
-3.87013,138.39503,326.3,0.39,0.36,"""2019-01-01""",410,"""N""","""VIIRS""","""n""",2,296.49,2.38,"""D""",0
-3.86444,138.56816,326.62,0.39,0.36,"""2019-01-01""",410,"""N""","""VIIRS""","""n""",2,290.5,2.46,"""D""",0
-3.79122,138.71609,334.24,0.39,0.36,"""2019-01-01""",410,"""N""","""VIIRS""","""n""",2,291.15,3.71,"""D""",0
-1.39299,132.29683,327.48,0.47,0.48,"""2019-01-01""",411,"""N""","""VIIRS""","""n""",2,292.37,2.06,"""D""",0
-2.43901,133.14156,335.7,0.43,0.46,"""2019-01-01""",411,"""N""","""VIIRS""","""n""",2,283.32,2.07,"""D""",2


In [7]:
# 5 baris sampel acak langsung dari CSV menggunakan DuckDB SAMPLE
duckdb.sql(f"SELECT * FROM read_csv_auto('{DATA_GLOB}', union_by_name=true) USING SAMPLE 5").show()

┌──────────┬───────────┬────────────┬────────┬────────┬────────────┬──────────┬───────────┬────────────┬────────────┬─────────┬────────────┬────────┬──────────┬───────┐
│ latitude │ longitude │ bright_ti4 │  scan  │ track  │  acq_date  │ acq_time │ satellite │ instrument │ confidence │ version │ bright_ti5 │  frp   │ daynight │ type  │
│  double  │  double   │   double   │ double │ double │    date    │ varchar  │  varchar  │  varchar   │  varchar   │  int64  │   double   │ double │ varchar  │ int64 │
├──────────┼───────────┼────────────┼────────┼────────┼────────────┼──────────┼───────────┼────────────┼────────────┼─────────┼────────────┼────────┼──────────┼───────┤
│ -2.75042 │ 105.96886 │     340.24 │    0.4 │   0.44 │ 2019-11-09 │ 0643     │ N         │ VIIRS      │ n          │       2 │     296.94 │   3.83 │ D        │     0 │
│ -4.24763 │ 105.17416 │     305.43 │    0.4 │   0.37 │ 2019-08-30 │ 1816     │ N         │ VIIRS      │ n          │       2 │      291.8 │   0.49 │ N    

## 7. Cek Missing Values (Nilai Kosong)

In [8]:
# Hitung missing value pada tiap kolom secara lazy
null_df = lf.select(pl.all().null_count()).collect()

hasil_null = []
for col in schema_polars.names():
    jml = null_df[col][0]
    persen = (jml / total_baris) * 100
    hasil_null.append({
        "kolom": col,
        "jumlah_null": jml,
        "persen_null (%)": round(persen, 4)
    })

df_cek_null = pl.DataFrame(hasil_null)
print(df_cek_null)

kolom_berisi_null = df_cek_null.filter(pl.col("jumlah_null") > 0).height
print(f"\nJumlah kolom yang memiliki missing value: {kolom_berisi_null} dari {total_kolom} kolom.")

shape: (15, 3)
┌────────────┬─────────────┬─────────────────┐
│ kolom      ┆ jumlah_null ┆ persen_null (%) │
│ ---        ┆ ---         ┆ ---             │
│ str        ┆ i64         ┆ f64             │
╞════════════╪═════════════╪═════════════════╡
│ latitude   ┆ 0           ┆ 0.0             │
│ longitude  ┆ 0           ┆ 0.0             │
│ bright_ti4 ┆ 0           ┆ 0.0             │
│ scan       ┆ 0           ┆ 0.0             │
│ track      ┆ 0           ┆ 0.0             │
│ …          ┆ …           ┆ …               │
│ version    ┆ 0           ┆ 0.0             │
│ bright_ti5 ┆ 0           ┆ 0.0             │
│ frp        ┆ 0           ┆ 0.0             │
│ daynight   ┆ 0           ┆ 0.0             │
│ type       ┆ 0           ┆ 0.0             │
└────────────┴─────────────┴─────────────────┘

Jumlah kolom yang memiliki missing value: 0 dari 15 kolom.


## 8. Cek Duplikasi Baris
Pengecekan dilakukan berdasarkan kombinasi kunci lokasi dan waktu: `latitude`, `longitude`, `acq_date`, dan `acq_time`.

In [9]:
kunci_duplikat = ["latitude", "longitude", "acq_date", "acq_time"]

# Hitung baris unik berdasarkan 4 kolom kunci tersebut
baris_unik = lf.select(kunci_duplikat).unique().select(pl.len()).collect().item()
kelebihan_duplikat = total_baris - baris_unik

print(f"Kombinasi kunci: {kunci_duplikat}")
print(f"Total baris data : {total_baris:,}")
print(f"Jumlah data unik : {baris_unik:,}")
print(f"Kelebihan duplikat: {kelebihan_duplikat:,} baris ({kelebihan_duplikat / total_baris * 100:.4f}%)")

Kombinasi kunci: ['latitude', 'longitude', 'acq_date', 'acq_time']
Total baris data : 1,011,936
Jumlah data unik : 1,011,936
Kelebihan duplikat: 0 baris (0.0000%)


## 9. Distribusi Kolom Kategori dan Pengecekan Kolom Konstan

In [10]:
# Tampilkan frekuensi nilai pada kolom kategori
kolom_kategori = ["confidence", "daynight", "satellite", "instrument", "version", "type"]

for k in kolom_kategori:
    print(f"=== Distribusi kolom: {k} ===")
    tabel = (
        lf.group_by(k)
        .len()
        .rename({"len": "jumlah"})
        .sort("jumlah", descending=True)
        .collect()
    )
    print(tabel)
    print()

=== Distribusi kolom: confidence ===


shape: (3, 2)
┌────────────┬────────┐
│ confidence ┆ jumlah │
│ ---        ┆ ---    │
│ str        ┆ u32    │
╞════════════╪════════╡
│ n          ┆ 909881 │
│ l          ┆ 69397  │
│ h          ┆ 32658  │
└────────────┴────────┘

=== Distribusi kolom: daynight ===
shape: (2, 2)
┌──────────┬────────┐
│ daynight ┆ jumlah │
│ ---      ┆ ---    │
│ str      ┆ u32    │
╞══════════╪════════╡
│ D        ┆ 696601 │
│ N        ┆ 315335 │
└──────────┴────────┘

=== Distribusi kolom: satellite ===
shape: (1, 2)
┌───────────┬─────────┐
│ satellite ┆ jumlah  │
│ ---       ┆ ---     │
│ str       ┆ u32     │
╞═══════════╪═════════╡
│ N         ┆ 1011936 │
└───────────┴─────────┘

=== Distribusi kolom: instrument ===


shape: (1, 2)
┌────────────┬─────────┐
│ instrument ┆ jumlah  │
│ ---        ┆ ---     │
│ str        ┆ u32     │
╞════════════╪═════════╡
│ VIIRS      ┆ 1011936 │
└────────────┴─────────┘

=== Distribusi kolom: version ===
shape: (1, 2)
┌─────────┬─────────┐
│ version ┆ jumlah  │
│ ---     ┆ ---     │
│ i64     ┆ u32     │
╞═════════╪═════════╡
│ 2       ┆ 1011936 │
└─────────┴─────────┘

=== Distribusi kolom: type ===
shape: (4, 2)
┌──────┬────────┐
│ type ┆ jumlah │
│ ---  ┆ ---    │
│ i64  ┆ u32    │
╞══════╪════════╡
│ 0    ┆ 938231 │
│ 2    ┆ 50347  │
│ 1    ┆ 18959  │
│ 3    ┆ 4399   │
└──────┴────────┘



In [11]:
# Cek kolom yang nilainya konstan (hanya 1 nilai unik di seluruh dataset)
# Hitung n_unique sekali saja untuk semua kolom secara efisien
df_n_unik = lf.select(pl.all().n_unique()).collect()

kolom_konstan = []
for col in schema_polars.names():
    if df_n_unik[col][0] == 1:
        nilai_pertama = lf.select(pl.col(col).first()).collect().item()
        kolom_konstan.append({"kolom": col, "nilai_konstan": str(nilai_pertama)})

if len(kolom_konstan) > 0:
    print("Ditemukan kolom bernilai konstan (hanya 1 nilai unik):")
    print(pl.DataFrame(kolom_konstan))
    print("Catatan: Kolom-kolom ini dapat dihapus saat data cleaning karena tidak memiliki variasi informasi.")
else:
    print("Tidak ada kolom bernilai konstan.")

Ditemukan kolom bernilai konstan (hanya 1 nilai unik):
shape: (3, 2)
┌────────────┬───────────────┐
│ kolom      ┆ nilai_konstan │
│ ---        ┆ ---           │
│ str        ┆ str           │
╞════════════╪═══════════════╡
│ satellite  ┆ N             │
│ instrument ┆ VIIRS         │
│ version    ┆ 2             │
└────────────┴───────────────┘
Catatan: Kolom-kolom ini dapat dihapus saat data cleaning karena tidak memiliki variasi informasi.


## 10. Statistik Dasar Kolom Numerik Penting

In [12]:
# Statistik deskriptif untuk kolom angka penting (suhu kecerahan dan daya radiasi api / FRP)
kolom_angka = ["bright_ti4", "bright_ti5", "frp", "scan", "track"]
lf.select(kolom_angka).collect().describe()

statistic,bright_ti4,bright_ti5,frp,scan,track
str,f64,f64,f64,f64,f64
"""count""",1.011936e6,1.011936e6,1.011936e6,1.011936e6,1.011936e6
"""null_count""",0.0,0.0,0.0,0.0,0.0
"""mean""",331.589812,294.494313,7.699509,0.455139,0.469923
"""std""",16.364937,8.676302,13.641358,0.082492,0.109067
"""min""",208.0,150.0,0.0,0.32,0.36
"""25%""",322.6,289.95,2.41,0.39,0.38
"""50%""",334.26,294.5,4.57,0.43,0.43
"""75%""",341.56,300.3,8.21,0.5,0.55
"""max""",367.0,380.61,1322.99,0.8,0.78


## 11. Profil Temporal (Rentang Waktu dan Deteksi per Tahun)

In [13]:
# Baca data dengan parsing tanggal otomatis
lf_tanggal = pl.scan_csv(DATA_GLOB, try_parse_dates=True)

info_tanggal = lf_tanggal.select([
    pl.col("acq_date").min().alias("tgl_awal"),
    pl.col("acq_date").max().alias("tgl_akhir"),
    pl.col("acq_date").n_unique().alias("hari_ada_data")
]).collect()

tgl_awal = info_tanggal["tgl_awal"][0]
tgl_akhir = info_tanggal["tgl_akhir"][0]
hari_ada = info_tanggal["hari_ada_data"][0]
total_hari_kalender = (tgl_akhir - tgl_awal).days + 1
hari_kosong = total_hari_kalender - hari_ada

print(f"Rentang Waktu      : {tgl_awal} s.d. {tgl_akhir}")
print(f"Total Hari Kalender: {total_hari_kalender} hari")
print(f"Hari Ada Deteksi   : {hari_ada} hari")
print(f"Hari Tanpa Deteksi : {hari_kosong} hari")

Rentang Waktu      : 2019-01-01 s.d. 2024-12-31
Total Hari Kalender: 2192 hari
Hari Ada Deteksi   : 2155 hari
Hari Tanpa Deteksi : 37 hari


In [14]:
# Mencari tanggal apa saja yang tidak ada rekaman deteksi (menggunakan himpunan / set standar python)
semua_tanggal = pl.date_range(tgl_awal, tgl_akhir, interval="1d", eager=True).to_list()
tanggal_tercatat = lf_tanggal.select(pl.col("acq_date").unique()).collect()["acq_date"].to_list()

list_hari_kosong = sorted(list(set(semua_tanggal) - set(tanggal_tercatat)))
print(f"Jumlah hari tanpa deteksi: {len(list_hari_kosong)} hari")
print("Contoh 10 hari kosong pertama:")
for t in list_hari_kosong[:10]:
    print(" -", t)

Jumlah hari tanpa deteksi: 37 hari
Contoh 10 hari kosong pertama:
 - 2022-07-27
 - 2022-07-28
 - 2022-07-29
 - 2022-07-30
 - 2022-07-31
 - 2022-08-01
 - 2022-08-02
 - 2022-08-03
 - 2022-08-04
 - 2022-08-05


In [15]:
# Rekapitulasi jumlah titik panas yang terdeteksi per tahun
deteksi_per_tahun = (
    lf_tanggal.group_by(pl.col("acq_date").dt.year().alias("tahun"))
    .len()
    .rename({"len": "jumlah_deteksi"})
    .sort("tahun")
    .collect()
)
deteksi_per_tahun

tahun,jumlah_deteksi
i32,u32
2019,437281
2020,91536
2021,68325
2022,57220
2023,253388
2024,104186


## 12. Profil Spasial (Batas Koordinat Wilayah Indonesia)

In [16]:
# Cek nilai batas koordinat latitude dan longitude
batas_spasial = lf.select([
    pl.col("latitude").min().alias("lat_min"),
    pl.col("latitude").max().alias("lat_max"),
    pl.col("longitude").min().alias("lon_min"),
    pl.col("longitude").max().alias("lon_max")
]).collect()

print("Sebaran batas koordinat pada data:")
print(batas_spasial)

Sebaran batas koordinat pada data:
shape: (1, 4)
┌───────────┬─────────┬──────────┬──────────┐
│ lat_min   ┆ lat_max ┆ lon_min  ┆ lon_max  │
│ ---       ┆ ---     ┆ ---      ┆ ---      │
│ f64       ┆ f64     ┆ f64      ┆ f64      │
╞═══════════╪═════════╪══════════╪══════════╡
│ -10.97676 ┆ 5.87273 ┆ 95.01992 ┆ 140.9774 │
└───────────┴─────────┴──────────┴──────────┘


In [17]:
# Cek apakah ada titik panas yang jatuh di luar kotak kasar Indonesia (Bujur 95-141, Lintang -11 sampai 6)
# Catatan: Batas kotak ini adalah perkiraan kasar sederhana, bukan batas resmi teritorial negara.
titik_di_luar = (
    lf.filter(
        (pl.col("longitude") < LON_MIN) | (pl.col("longitude") > LON_MAX) |
        (pl.col("latitude") < LAT_MIN) | (pl.col("latitude") > LAT_MAX)
    )
    .select(pl.len())
    .collect()
    .item()
)

print(f"Jumlah titik di luar kotak kasar Indonesia: {titik_di_luar} titik")
if titik_di_luar == 0:
    print("Semua koordinat titik panas berada dalam kotak area Indonesia.")

Jumlah titik di luar kotak kasar Indonesia: 0 titik
Semua koordinat titik panas berada dalam kotak area Indonesia.


## 13. Profiling dengan DuckDB SUMMARIZE

In [18]:
# Menggunakan fitur SUMMARIZE DuckDB langsung dari file CSV tanpa memuat seluruh data ke RAM
duckdb.sql(f"SUMMARIZE SELECT * FROM read_csv_auto('{DATA_GLOB}', union_by_name=true)").show(max_width=250, max_rows=20)

┌─────────────┬─────────────┬────────────┬────────────┬───────────────┬────────────────────────────┬─────────────────────┬────────────────────┬─────────────────────┬─────────────────────┬─────────┬─────────────────┐
│ column_name │ column_type │    min     │    max     │ approx_unique │            avg             │         std         │        q25         │         q50         │         q75         │  count  │ null_percentage │
│   varchar   │   varchar   │  varchar   │  varchar   │     int64     │          varchar           │       varchar       │      varchar       │       varchar       │       varchar       │  int64  │  decimal(9,2)   │
├─────────────┼─────────────┼────────────┼────────────┼───────────────┼────────────────────────────┼─────────────────────┼────────────────────┼─────────────────────┼─────────────────────┼─────────┼─────────────────┤
│ latitude    │ DOUBLE      │ -10.97676  │ 5.87273    │        676898 │ -3.4486164680276086        │ 3.5311396849922185  │ -7.0378185399

## 14. Cek Silang (Cross-Check): Total Baris Polars vs DuckDB

In [19]:
# Verifikasi kebenaran pembacaan data antara Polars dan DuckDB
baris_duckdb = duckdb.sql(f"SELECT COUNT(*) FROM read_csv_auto('{DATA_GLOB}', union_by_name=true)").fetchone()[0]

print(f"Total baris (Polars) : {total_baris:,}")
print(f"Total baris (DuckDB) : {baris_duckdb:,}")

if total_baris == baris_duckdb:
    print("Hasil: Keduanya sama persis (COCOK).")
else:
    print("Hasil: Terdapat perbedaan jumlah baris!")

Total baris (Polars) : 1,011,936
Total baris (DuckDB) : 1,011,936
Hasil: Keduanya sama persis (COCOK).


## 15. Simpan Hasil Gabungan ke Parquet (Lazy Sink)

In [20]:
# Buat folder tujuan bila belum ada
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
file_parquet = PROCESSED_DIR / "firms_viirs_snpp_indonesia_raw.parquet"

# Simpan lazy frame langsung ke file Parquet (hemat RAM dan cepat)
lf.sink_parquet(file_parquet)

ukuran_parquet_mb = file_parquet.stat().st_size / (1024 * 1024)
print(f"File Parquet tersimpan di : {file_parquet}")
print(f"Ukuran file Parquet      : {ukuran_parquet_mb:.2f} MB (Jauh lebih hemat dibanding CSV asli {total_ukuran_mb:.2f} MB)")

File Parquet tersimpan di : ..\data\processed\firms_viirs_snpp_indonesia_raw.parquet
Ukuran file Parquet      : 17.65 MB (Jauh lebih hemat dibanding CSV asli 76.37 MB)


## 16. Tabel Ringkasan Hasil Profiling Data

In [21]:
# Tabel rangkuman untuk bahan dokumentasi dan data/README.md
df_rekap = pl.DataFrame({
    "Indikator": [
        "Jumlah File CSV",
        "Total Ukuran CSV (MB)",
        "Total Baris Data",
        "Jumlah Kolom",
        "Syarat > 1 Juta Baris",
        "Syarat >= 500 MB",
        "Rentang Waktu",
        "Hari Ada Data / Total Hari",
        "Hari Tanpa Data",
        "Kolom dengan Missing Value",
        "Baris Duplikat",
        "Titik di Luar Kotak Indonesia",
        "Cek Silang Polars vs DuckDB"
    ],
    "Hasil Profiling": [
        f"{len(files)} file",
        f"{total_ukuran_mb:.2f} MB",
        f"{total_baris:,} baris",
        f"{total_kolom} kolom",
        "Ya (Lolos)",
        f"Tidak ({total_ukuran_mb:.2f} MB < 500 MB)",
        f"{tgl_awal} s.d. {tgl_akhir}",
        f"{hari_ada} / {total_hari_kalender} hari",
        f"{hari_kosong} hari",
        f"{kolom_berisi_null} kolom",
        f"{kelebihan_duplikat} baris",
        f"{titik_di_luar} titik",
        f"Sama persis ({total_baris:,} baris)"
    ]
})
df_rekap

Indikator,Hasil Profiling
str,str
"""Jumlah File CSV""","""6 file"""
"""Total Ukuran CSV (MB)""","""76.37 MB"""
"""Total Baris Data""","""1,011,936 baris"""
"""Jumlah Kolom""","""15 kolom"""
"""Syarat > 1 Juta Baris""","""Ya (Lolos)"""
…,…
"""Hari Tanpa Data""","""37 hari"""
"""Kolom dengan Missing Value""","""0 kolom"""
"""Baris Duplikat""","""0 baris"""


## 17. AI Disclosure Statement

> **Alat AI yang digunakan:** Claude (Anthropic) melalui asisten pengerjaan tugas.
>
> **Bagian yang dibantu:**
> - Membantu menyusun alur notebook data profiling dari data mentah.
> - Membantu sintaks fungsi lazy pada Polars (`scan_csv`, `sink_parquet`) dan DuckDB `SUMMARIZE`.
> - Menjalankan eksekusi awal dan menyusun draf temuan angka-angka profiling.
>
> **Verifikasi yang dilakukan:** 
>
> Saya menjalankan ulang seluruh notebook dari awal setelah restart kernel. Saya memverifikasi ulang cell per cell dan mencocokkan jumlah baris,  ukuran file, dan rentang tanggal . Saya menelusuri sampel baris ke file CSV mentah, dan mencocokkan arti kolom. Interpretasi dan kesimpulan ditulis berdasarkan output.
>
>